# Dataset v2: compact TCN backend, three seeds

Этот controlled experiment меняет только temporal backend:

```text
frozen ImageNet ResNet18 frame features [24, 512]
-> compact residual TCN
-> word classifier
```

Он сравнивается с уже выполненным `BiGRU` на том же dataset v2, с тем же
speaker-word balanced sampler и seed `42`, `123`, `2026`.

Это TCN backend ablation, а не полный LRW-style model: полный вариант также
требует trainable/pretrained 3D frontend и lip-reading visual encoder.
Архитектурная мотивация: [официальный TCN lip-reading repository](https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks).
Test split не загружается и не используется.


## 1. Подключить Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Загрузить frozen features dataset v2

In [ ]:
from collections import Counter
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
feature_root = drive_root / 'controlled_frozen_resnet18_bigru_balanced_v2'
train_cache_path = feature_root / 'dataset_v2_train_resnet18_imagenet_features.pt'
val_cache_path = feature_root / 'dataset_v2_val_resnet18_imagenet_features.pt'

if not train_cache_path.exists() or not val_cache_path.exists():
    raise FileNotFoundError(
        'Dataset v2 feature caches were not found. Run the balanced v2 notebook first.'
    )

train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)

word_label_pairs = set(zip(train_cache['words'], train_cache['labels'].tolist()))
word_to_idx = {word: int(label) for word, label in word_label_pairs}
idx_to_word = {index: word for word, index in word_to_idx.items()}
vocab = [idx_to_word[index] for index in sorted(idx_to_word)]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
output_dir = drive_root / 'controlled_tcn_backend_v2'
output_dir.mkdir(parents=True, exist_ok=True)

bigru_runs_path = (
    drive_root / 'balanced_multiseed_v1_v2' / 'v1_v2_multiseed_runs.csv'
)

print('device:', device)
print('train features:', train_cache['features'].shape)
print('validation features:', val_cache['features'].shape)
print('train speakers:', sorted(set(train_cache['speakers'])))
print('validation speakers:', sorted(set(val_cache['speakers'])))
print('vocab:', vocab)
print('results:', output_dir)
print('BiGRU comparison found:', bigru_runs_path.exists())
print('TEST SPLIT IS NOT LOADED.')


## 3. Проверить feature cache и class targets

In [ ]:
assert train_cache['features'].shape[1:] == (24, 512)
assert val_cache['features'].shape[1:] == (24, 512)
assert len(train_cache['features']) == 1100
assert len(val_cache['features']) == 105
assert set(train_cache['words']) == set(vocab)
assert set(val_cache['words']) == set(vocab)
assert set(train_cache['speakers']).isdisjoint(set(val_cache['speakers']))

round_trip_ok = all(
    idx_to_word[word_to_idx[word]] == word
    for word in vocab
)
print('word -> class id -> word:', round_trip_ok)
print('non-empty speaker-word groups:', len(set(zip(
    train_cache['speakers'], train_cache['words']
))))
assert round_trip_ok


## 4. Balanced Dataset, compact residual TCN и метрики

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


class FeatureSequenceDataset(Dataset):
    def __init__(self, cache, word_to_idx):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']],
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class TemporalResidualBlock(nn.Module):
    def __init__(self, channels, dilation, dropout=0.3):
        super().__init__()
        padding = dilation
        self.block = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                kernel_size=3,
                padding=padding,
                dilation=dilation,
            ),
            nn.GroupNorm(8, channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(
                channels,
                channels,
                kernel_size=3,
                padding=padding,
                dilation=dilation,
            ),
            nn.GroupNorm(8, channels),
        )
        self.activation = nn.ReLU()

    def forward(self, features):
        return self.activation(features + self.block(features))


class CompactTCNClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, channels=128, dropout=0.3):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=1),
            nn.GroupNorm(8, channels),
            nn.ReLU(),
        )
        self.temporal_blocks = nn.Sequential(
            TemporalResidualBlock(channels, dilation=1, dropout=dropout),
            TemporalResidualBlock(channels, dilation=2, dropout=dropout),
            TemporalResidualBlock(channels, dilation=4, dropout=dropout),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(channels, num_classes),
        )

    def forward(self, features):
        # Input: [batch, frames, 512]. Conv1d expects [batch, channels, frames].
        features = features.transpose(1, 2)
        features = self.input_projection(features)
        features = self.temporal_blocks(features)
        return self.classifier(features)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def make_loaders(train_cache, val_cache, seed):
    group_keys = list(zip(train_cache['speakers'], train_cache['words']))
    group_counts = Counter(group_keys)
    sample_weights = torch.tensor(
        [1.0 / group_counts[key] for key in group_keys],
        dtype=torch.double,
    )
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(seed),
    )
    train_loader = DataLoader(
        FeatureSequenceDataset(train_cache, word_to_idx),
        batch_size=32,
        sampler=sampler,
        num_workers=0,
    )
    val_loader = DataLoader(
        FeatureSequenceDataset(val_cache, word_to_idx),
        batch_size=32,
        shuffle=False,
        num_workers=0,
    )
    _ = next(iter(train_loader))
    return train_loader, val_loader, len(group_counts)


def classification_metrics(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1

    class_f1 = []
    class_recall = []
    for class_id in range(num_classes):
        true_positive = matrix[class_id, class_id]
        support = matrix[class_id, :].sum()
        predicted = matrix[:, class_id].sum()
        precision = true_positive / predicted if predicted else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        if support:
            class_f1.append(f1)
            class_recall.append(recall)

    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean(class_f1)),
        'balanced_accuracy': float(np.mean(class_recall)),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            total_loss += criterion(outputs, batch_y).item() * batch_y.size(0)
            total += batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())
    metrics = classification_metrics(preds, labels, len(vocab))
    metrics['loss'] = total_loss / total
    return metrics


set_reproducible_seed(42)
preview_model = CompactTCNClassifier(num_classes=len(vocab)).to(device)
print('trainable TCN parameters:', count_parameters(preview_model))


## 5. Одинаковый training protocol для одного TCN-запуска

In [ ]:
def run_one_seed(seed, num_epochs=30, patience=7):
    set_reproducible_seed(seed)
    train_loader, val_loader, group_count = make_loaders(train_cache, val_cache, seed)
    set_reproducible_seed(seed)
    model = CompactTCNClassifier(num_classes=len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    best_row = None
    best_state = None
    no_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * batch_y.size(0)
            train_correct += (outputs.argmax(dim=1) == batch_y).sum().item()
            train_total += batch_y.size(0)

        val_metrics = evaluate(model, val_loader, criterion)
        train_loss = train_loss_sum / train_total
        train_accuracy = train_correct / train_total
        improved = (
            val_metrics['macro_f1'] > best_macro_f1
            or (
                val_metrics['macro_f1'] == best_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )

        if improved:
            best_macro_f1 = val_metrics['macro_f1']
            best_val_loss = val_metrics['loss']
            no_improvement = 0
            best_row = {
                'model': 'Compact residual TCN on frozen ResNet18 features',
                'dataset_version': 'v2',
                'seed': seed,
                'best_epoch': epoch,
                'train_clips': len(train_cache['labels']),
                'speaker_word_groups': group_count,
                'train_accuracy_at_best': train_accuracy,
                'train_loss_at_best': train_loss,
                'validation_loss': val_metrics['loss'],
                'validation_accuracy': val_metrics['accuracy'],
                'validation_macro_f1': val_metrics['macro_f1'],
                'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
                'predicted_classes': val_metrics['predicted_classes'],
                'trainable_parameters': count_parameters(model),
            }
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            no_improvement += 1

        if no_improvement >= patience:
            break

    elapsed = time.perf_counter() - start
    best_row['epochs_completed'] = epoch
    best_row['training_seconds'] = elapsed
    checkpoint_path = output_dir / f'tcn_v2_seed{seed}_best.pt'
    torch.save({
        'model_state_dict': best_state,
        'vocab': vocab,
        'result': best_row,
        'test_used': False,
    }, checkpoint_path)

    print(
        f"TCN v2 seed={seed} | epoch={best_row['best_epoch']} | "
        f"acc={best_row['validation_accuracy']:.4f} | "
        f"macro-F1={best_row['validation_macro_f1']:.4f} | "
        f"balanced_acc={best_row['validation_balanced_accuracy']:.4f} | "
        f"classes={best_row['predicted_classes']}/10 | {elapsed:.1f}s"
    )
    return best_row


## 6. Запустить TCN на трёх seed

In [ ]:
SEEDS = [42, 123, 2026]
results = [run_one_seed(seed) for seed in SEEDS]

print('completed TCN runs:', len(results))
print('TEST SPLIT WAS NOT USED.')


## 7. Сравнить TCN с BiGRU на dataset v2

In [ ]:
tcn_runs = pd.DataFrame(results).sort_values('seed')
tcn_runs_path = output_dir / 'tcn_v2_multiseed_runs.csv'
tcn_runs.to_csv(tcn_runs_path, index=False)
display(tcn_runs)

metric_columns = [
    'validation_accuracy',
    'validation_macro_f1',
    'validation_balanced_accuracy',
    'predicted_classes',
]
tcn_summary = tcn_runs[metric_columns].agg(['mean', 'std']).T
tcn_summary.columns = ['tcn_mean', 'tcn_std']

comparison = None
if bigru_runs_path.exists():
    bigru_all_runs = pd.read_csv(bigru_runs_path)
    bigru_runs = bigru_all_runs[
        bigru_all_runs['dataset_version'] == 'v2'
    ].copy()
    bigru_summary = bigru_runs[metric_columns].agg(['mean', 'std']).T
    bigru_summary.columns = ['bigru_mean', 'bigru_std']
    comparison = bigru_summary.join(tcn_summary).reset_index(names='metric')
    comparison['tcn_minus_bigru_mean'] = (
        comparison['tcn_mean'] - comparison['bigru_mean']
    )
    display(comparison)

    paired = bigru_runs.set_index('seed')[['validation_macro_f1']].rename(
        columns={'validation_macro_f1': 'bigru_macro_f1'}
    ).join(
        tcn_runs.set_index('seed')[['validation_macro_f1']].rename(
            columns={'validation_macro_f1': 'tcn_macro_f1'}
        )
    )
    paired['tcn_minus_bigru_macro_f1'] = (
        paired['tcn_macro_f1'] - paired['bigru_macro_f1']
    )
    display(paired)
    paired.to_csv(output_dir / 'tcn_vs_bigru_paired_macro_f1.csv')

    figure_path = output_dir / 'tcn_vs_bigru_macro_f1.png'
    fig, ax = plt.subplots(figsize=(7, 4))
    for seed, row in paired.iterrows():
        ax.plot(
            ['BiGRU', 'TCN'],
            [row['bigru_macro_f1'], row['tcn_macro_f1']],
            marker='o',
            label=f'seed {seed}',
        )
    ax.set_ylabel('validation macro-F1')
    ax.set_title('Dataset v2: temporal backend comparison')
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('BiGRU multi-seed CSV was not found:', bigru_runs_path)

tcn_summary_path = output_dir / 'tcn_v2_multiseed_summary.csv'
tcn_summary.reset_index(names='metric').to_csv(tcn_summary_path, index=False)
print('saved TCN runs:', tcn_runs_path)
print('saved TCN summary:', tcn_summary_path)


## 8. Decision gate

In [ ]:
tcn_mean = tcn_runs['validation_macro_f1'].mean()
print('TCN mean macro-F1:', round(tcn_mean, 4))

if bigru_runs_path.exists():
    bigru_mean = bigru_runs['validation_macro_f1'].mean()
    delta = tcn_mean - bigru_mean
    wins = int((paired['tcn_minus_bigru_macro_f1'] > 0).sum())
    print('BiGRU mean macro-F1:', round(bigru_mean, 4))
    print('mean TCN - BiGRU:', round(delta, 4))
    print('TCN wins by seed:', f'{wins}/{len(SEEDS)}')

    if delta > 0 and wins >= 2:
        print('DECISION: TCN backend is the better controlled temporal model.')
    else:
        print('DECISION: TCN backend alone does not solve the bottleneck.')
        print('Next: use a lip-reading/video-pretrained visual frontend.')

print('TEST SPLIT WAS NOT USED.')
